# Modul B · Kapitel 1.5 — Tree-of-Thought

## Challenge: Mehrere Lösungszweige erzeugen, bewerten und auswählen


**Lernziel:** Du setzt Tree-of-Thought aus vorbereiteten Bausteinen zusammen und vergleichst das Ergebnis mit einem normalen Prompt und Self-Consistency.

### So funktioniert dieses Notebook

| Symbol | Bedeutung |
|:--:|---|
| 📖 | Erklärung — lesen |
| ▶️ | Fertiger Code — einfach ausführen (`Shift` + `Enter`) |
| 🛠️ | **Challenge** — hier schreibst du selbst Code |
| ✅ | Selbsttest — sagt dir sofort, ob deine Lösung stimmt |
| 💡 | **Lösung** — zum Aufklappen, wenn du nicht weiterkommst |

Es gibt genau **eine Challenge**: die Tree-of-Thought-Schleife implementieren.


---
## 0 · Setup

▶️ Das Setup lädt einen erfundenen Incident-Report und fünf vorbereitete Prüfpunkte. Die Prüfpunkte machen die drei Verfahren am Ende vergleichbar.

`MAX_ANTWORT_TOKENS = 2400` ist eine Obergrenze für die erzeugte Antwort. Kurze Antworten enden vorher. Das größere Budget verhindert, dass ein längerer Plan vor seinem festen Endformat abgeschnitten wird.


In [ ]:
# ▶️ Pakete, Pfade und Modellzugang
import re
import sys
from collections import Counter
from pathlib import Path

try:
    import openai
except ImportError:
    %pip install -q openai
    import openai

for kandidat in [Path.cwd(), *Path.cwd().parents, Path("/content"),
                  Path("/content/01_prompt-engineering")]:
    if (kandidat / "helfer.py").exists():
        sys.path.insert(0, str(kandidat))
        break

from helfer import BASIS_URL, MODELL, frage_llm, lade_daten, zeige

MAX_ANTWORT_TOKENS = 2400
print(f"Server: {BASIS_URL}")
print(f"Modell: {MODELL}")


In [ ]:
# ▶️ Aufgabe und vorbereitete Bewertung
IR = lade_daten("incident_report")
STICHWORTE = lade_daten("05_abdeckung")
AUFGABE = f"# Incident report\n{IR['report']}\n\n# Question\n{IR['frage']}"


def pruefe_abdeckung(text):
    """Prüft fünf sichtbare Mindestanforderungen mit einfachen Textregeln."""
    klein = text.lower()
    punkte = list(IR["pruefpunkte"])
    betroffen = (
        "srv-pam01" in klein and "cve-2026-3224" in klein
        and "srv-pam01 is unaffected" not in klein
    )
    account = (
        "svc_backup" in klein
        and any(wort in klein for wort in ["compromis", "disable", "revoke", "reset", "remove"])
    )
    zugangsdaten = (
        "vault" in klein
        and any(wort in klein for wort in ["rotat", "revoke", "reset", "disable", "remove", "password"])
    )
    share = "srv-fs02" in klein and any(
        wort in klein for wort in [".locked", "encrypt", "share"]
    )
    reihenfolge = any(
        wort in klein for wort in ["first:", "then", "next", "immediately", "1."]
    )
    return dict(zip(punkte, [betroffen, account, zugangsdaten, share, reihenfolge]))


def abdeckung(text):
    treffer = pruefe_abdeckung(text)
    return sum(treffer.values()) / len(treffer)


def lies_first(text):
    treffer = re.search(r"FIRST\s*:\s*([A-Za-z0-9_.-]+)", text, re.IGNORECASE)
    return treffer.group(1).lower().rstrip(".,;:") if treffer else None


assert list(STICHWORTE) == IR["pruefpunkte"]
print(f"Incident: {IR['id']} — {IR['titel']}")
print(f"{len(IR['pruefpunkte'])} Prüfpunkte für den Vergleich geladen.")


---
## 1 · Die Aufgabe

📖 Ein Incident betrifft einen Credential-Vault, einen kompromittierten Service-Account und einen verschlüsselten File-Server. Die Aufgabe lautet:

> Welche Systeme müssen zuerst vom Netz, und in welcher Reihenfolge soll das Team während der nächsten vier Stunden handeln?

Anders als beim Zählen gibt es mehrere vertretbare Pläne. Ein Plan kann den Vault zuerst isolieren, ein anderer den File-Server. Entscheidend ist, ob die relevanten Systeme, Zugangsdaten und die Reihenfolge berücksichtigt werden. Genau hier ist die Auswahl zwischen mehreren Lösungszweigen sinnvoll.


In [ ]:
# ▶️ Report und Frage ansehen
zeige(IR["report"], titel="Incident-Report")
zeige(IR["frage"], titel="Aufgabe")

print("\nPrüfpunkte")
for punkt in IR["pruefpunkte"]:
    print("•", punkt)


---
## 2 · Gegeben: normaler Prompt und Self-Consistency

📖 Der normale Prompt erzeugt genau einen Plan. Das feste Feld `FIRST:` macht sichtbar, welches System der Plan zuerst isoliert.


In [ ]:
# ▶️ Normaler Prompt: ein Plan, ein Modellaufruf
def baue_plan_prompt(aufgabe):
    return f"""You are the incident commander.
Create one response plan for the incident below. WKS-114 is already isolated.
FIRST must be srv-pam01 or srv-fs02, never WKS-114.
Start with exactly: FIRST: <host name>
Then give four numbered actions for the next four hours.
Name concrete hosts and accounts. Stay under 140 words.

{aufgabe}"""


NORMALER_PROMPT = baue_plan_prompt(AUFGABE)
ANTWORT_NORMAL = frage_llm(
    NORMALER_PROMPT, temperature=0.0, max_tokens=MAX_ANTWORT_TOKENS
)
zeige(ANTWORT_NORMAL, titel="Normaler Prompt")


📖 Self-Consistency erzeugt drei vollständige Pläne. Die Hostnamen hinter `FIRST:` stimmen ab. Anschließend wird der erste Plan übernommen, der mit dem Mehrheitsentscheid beginnt.

Das Verfahren bewertet keine Zwischenstände. Es fragt nur: **Auf welchen ersten Schritt einigen sich mehrere vollständige Pfade?**


In [ ]:
# ▶️ Self-Consistency ist vollständig gegeben
def self_consistency(aufgabe, k=3):
    prompt = baue_plan_prompt(aufgabe)
    antworten = [
        frage_llm(prompt, temperature=0.8, max_tokens=MAX_ANTWORT_TOKENS)
        for _ in range(k)
    ]
    erste_schritte = [lies_first(text) for text in antworten]
    stimmen = Counter(wert for wert in erste_schritte if wert is not None)
    gewinner = stimmen.most_common(1)[0][0] if stimmen else None
    antwort = next(
        (text for text, erster in zip(antworten, erste_schritte) if erster == gewinner),
        antworten[0],
    )
    return {"antwort": antwort, "stimmen": erste_schritte, "gewinner": gewinner}


ERGEBNIS_SC = self_consistency(AUFGABE, k=3)
print("FIRST-Stimmen:", ERGEBNIS_SC["stimmen"])
print("Mehrheit:", ERGEBNIS_SC["gewinner"])
zeige(ERGEBNIS_SC["antwort"], titel="Self-Consistency")


---
## 3 · Tree-of-Thought

📖 Tree-of-Thought trifft die Auswahl früher und expliziter:

```text
Aufgabe
  ├─ Kandidat 1 ─ Bewertung
  ├─ Kandidat 2 ─ Bewertung
  └─ Kandidat 3 ─ Bewertung
                         ↓
                 bester Kandidat
```

Die Bausteine sind gegeben:

- `erzeuge_ansaetze()` erzeugt drei unterschiedliche Pläne,
- `bewerte_ansatz()` bewertet jeden Plan gegen dieselben fünf Kriterien,
- `tree_of_thought()` soll anschließend den besten Zweig auswählen.

Das ist ein Baum mit einer Entscheidungsebene. Mehr Tiefe würde den gewählten Plan erneut verzweigen. Für das Grundprinzip reicht eine Ebene: **verzweigen, bewerten, auswählen**.


In [ ]:
# ▶️ Vorbereiteter Baustein 1: mehrere Zweige erzeugen
def erzeuge_ansaetze(aufgabe, n=3):
    prompt = f"""You are the incident commander.
Propose ONE response plan. This is one candidate branch, not a list of alternatives.
WKS-114 is already isolated. FIRST must be srv-pam01 or srv-fs02, never WKS-114.
Start with exactly: FIRST: <host name>
Then give four numbered actions. Name concrete hosts and accounts.
Stay under 140 words and do not repeat the report.

{aufgabe}"""
    ansaetze = []
    for nummer in range(1, n + 1):
        ansaetze.append(frage_llm(
            prompt, temperature=0.9, max_tokens=MAX_ANTWORT_TOKENS
        ))
        print(f"Kandidat {nummer}/{n} erzeugt", flush=True)
    return ansaetze


def lies_score(text, maximum=5):
    treffer = re.search(r"SCORE\s*:\s*(\d+)", text, re.IGNORECASE)
    if not treffer:
        return 0
    return min(int(treffer.group(1)), maximum)


In [ ]:
# ▶️ Vorbereiteter Baustein 2: einen Zweig bewerten
KRITERIEN_TEXT = "\n".join(
    f"{nummer}. {punkt}" for nummer, punkt in enumerate(IR["pruefpunkte"], start=1)
)


def bewerte_ansatz(aufgabe, ansatz):
    prompt = f"""Evaluate the proposed response plan against the five criteria.
Award one point only when a criterion is explicitly satisfied.
Return exactly two lines:
SCORE: <0-5>
REASON: <one short sentence>

# Criteria
{KRITERIEN_TEXT}

{aufgabe}

# Proposed plan
{ansatz}"""
    bewertung = frage_llm(prompt, temperature=0.0, max_tokens=MAX_ANTWORT_TOKENS)
    erster = lies_first(ansatz)
    if erster not in {"srv-pam01", "srv-fs02"}:
        return {
            "score": 0,
            "text": f"SCORE: 0/5\nREASON: FIRST {erster!r} is invalid because that host is already isolated.",
        }
    return {"score": lies_score(bewertung), "text": bewertung}


### 🛠️ Challenge 1: Tree-of-Thought zusammensetzen

Implementiere `tree_of_thought(aufgabe, breite=3)`.

Die Funktion soll:

1. mit `erzeuge_ansaetze(aufgabe, n=breite)` die Zweige erzeugen,
2. jeden Zweig mit `bewerte_ansatz(aufgabe, ansatz)` bewerten,
3. den Index mit dem höchsten `score` bestimmen,
4. ein Dictionary mit `antwort`, `ansaetze`, `bewertungen` und `bester_index` zurückgeben.

Bei Gleichstand gewinnt der erste Kandidat. `max(..., key=...)` erfüllt diese Regel automatisch.


In [ ]:
def tree_of_thought(aufgabe, breite=3):
    """Erzeugt, bewertet und wählt mehrere Lösungszweige."""
    # TODO: Zweige erzeugen, bewerten und den besten auswählen.
    raise NotImplementedError("Challenge 1: Tree-of-Thought implementieren")


In [ ]:
# ✅ Selbsttest — erzeugt und bewertet drei Kandidaten
ERGEBNIS_TOT = tree_of_thought(AUFGABE, breite=3)

assert set(ERGEBNIS_TOT) == {"antwort", "ansaetze", "bewertungen", "bester_index"}
assert len(ERGEBNIS_TOT["ansaetze"]) == 3
assert len(ERGEBNIS_TOT["bewertungen"]) == 3
assert 0 <= ERGEBNIS_TOT["bester_index"] < 3
assert ERGEBNIS_TOT["antwort"] == ERGEBNIS_TOT["ansaetze"][ERGEBNIS_TOT["bester_index"]]
assert all(0 <= b["score"] <= 5 for b in ERGEBNIS_TOT["bewertungen"])

print("✅ Challenge 1 gelöst")
for i, (ansatz, bewertung) in enumerate(
        zip(ERGEBNIS_TOT["ansaetze"], ERGEBNIS_TOT["bewertungen"]), start=1):
    marke = " ← gewählt" if i - 1 == ERGEBNIS_TOT["bester_index"] else ""
    print(f"Kandidat {i}: {bewertung['score']}/5{marke} · FIRST: {lies_first(ansatz)}")


<details>
<summary>💡 Lösung aufklappen — erst selbst probieren!</summary>

```python
def tree_of_thought(aufgabe, breite=3):
    ansaetze = erzeuge_ansaetze(aufgabe, n=breite)
    bewertungen = []
    for nummer, ansatz in enumerate(ansaetze, start=1):
        bewertungen.append(bewerte_ansatz(aufgabe, ansatz))
        print(f"Kandidat {nummer}/{breite} bewertet", flush=True)

    bester_index = max(
        range(len(ansaetze)), key=lambda i: bewertungen[i]["score"]
    )
    return {
        "antwort": ansaetze[bester_index],
        "ansaetze": ansaetze,
        "bewertungen": bewertungen,
        "bester_index": bester_index,
    }
```

</details>


---
## 4 · Vergleich der drei Verfahren

📖 Alle drei Verfahren beantworten dieselbe Aufgabe. Die vorbereitete Bewertungsfunktion prüft anschließend dieselben fünf erwarteten Inhalte. So vergleichen wir nicht Schreibstil, sondern sichtbare fachliche Abdeckung.


In [ ]:
# ▶️ Gegenüberstellung nach denselben Prüfpunkten
ANTWORTEN = {
    "Normal": ANTWORT_NORMAL,
    "Self-Consistency": ERGEBNIS_SC["antwort"],
    "Tree-of-Thought": ERGEBNIS_TOT["antwort"],
}
AUFRUFE = {"Normal": 1, "Self-Consistency": 3, "Tree-of-Thought": 6}
PRUEFUNGEN = {name: pruefe_abdeckung(text) for name, text in ANTWORTEN.items()}

print(f"{'Prüfpunkt':<62} {'Normal':>8} {'Self-Cons.':>12} {'Tree-of-T.':>12}")
print("─" * 98)
for punkt in IR["pruefpunkte"]:
    werte = ["✔" if PRUEFUNGEN[name][punkt] else "✘" for name in ANTWORTEN]
    print(f"{punkt:<62} {werte[0]:>8} {werte[1]:>12} {werte[2]:>12}")

print("─" * 98)
print(f"{'Abdeckung':<62} " + " ".join(
    f"{abdeckung(ANTWORTEN[name]):>11.0%}" for name in ANTWORTEN
))
print(f"{'Modellaufrufe':<62} " + " ".join(
    f"{AUFRUFE[name]:>11}" for name in ANTWORTEN
))


In [ ]:
# ▶️ Die gewählte Tree-of-Thought-Antwort ansehen
zeige(ERGEBNIS_TOT["antwort"], titel="Gewählter Tree-of-Thought-Zweig")
print("\nBewertung des Auswahlmodells:")
print(ERGEBNIS_TOT["bewertungen"][ERGEBNIS_TOT["bester_index"]]["text"])


📖 **Auswertung:** Mehr Modellaufrufe garantieren keine bessere Antwort. Self-Consistency nutzt Übereinstimmung zwischen vollständigen Plänen. Tree-of-Thought bewertet Kandidaten vor der Auswahl. Der Baum hilft nur, wenn die Bewertung tatsächlich nach der Qualität sortiert, die später zählt.

Die Textprüfung ist bewusst einfach und transparent. Sie verlangt bei jedem Prüfpunkt mehrere passende Signale und fängt die konkrete Negation `srv-pam01 is unaffected` ab. Vollständiges fachliches Verständnis ersetzt sie trotzdem nicht. Das Ergebnis beschreibt diesen einen Incident und diesen Modelllauf.


---
## 5 · Was du gebaut hast

- Der normale Prompt erzeugt einen einzelnen Plan.
- Self-Consistency lässt drei vollständige Pläne über den ersten Schritt abstimmen.
- Tree-of-Thought erzeugt drei Zweige, bewertet jeden und wählt den höchsten Score.
- Eine gemeinsame Kriterien-Tabelle vergleicht die drei Endantworten.

Der entscheidende Unterschied lautet: Self-Consistency entscheidet **nach mehreren vollständigen Pfaden per Mehrheit**. Tree-of-Thought entscheidet **zwischen explizit bewerteten Zweigen**.
